# M31 — Understand LLM Training

**Objective:** map the stages and objectives that produce a language model.

M30 already traces one transformer block. The useful whole here is the
**training system around that block**:

`tokens → window → shifted (input, target) pairs → next-token NLL → checkpoint → protected eval`

Causal next-token prediction scores the **following** token. A tiny
NumPy bigram table is the teaching stand-in. It is **not a production**
trainer. Inference stays closed (M32).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a pair table, a target count, a loss
inequality, a split overlap, or a stage label.

Do not open a decoder. Do not treat falling train loss as task quality.
The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M31" / "llm_training_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M27.tokenization_core import load_tokenizer
from missions.M30.transformer_block import (
    PRE_NORM_DIAGRAM,
    X_CASH_CONTEXT,
    observability_report as block_observability,
    transformer_block,
)
from missions.M31.llm_training_core import (
    CONTEXT_SEQUENCE,
    DATASET_VERSION,
    LEAK_DOC_ID,
    NLL_LOGITS,
    NLL_TARGET,
    SCALE_LIMIT,
    SHIFT_INPUTS,
    SHIFT_TARGETS,
    SHIFT_TOKENS,
    SHORT_CONTEXT_LENGTH,
    STAGE_DEFINITIONS,
    SYSTEM_MAP,
    TEACHING_CONTEXT_LENGTH,
    TRAINING_TIME_BOUNDARY,
    TRAINING_VERSION,
    classify_intervention,
    context_length_effect,
    first_divergence,
    frozen_block_logits,
    lineage_report,
    lineage_with_leak,
    load_teaching_corpus,
    load_teaching_tokenizer,
    n_prediction_targets,
    observability_report,
    pipeline_with_defect,
    repair_run,
    run_causal_pipeline,
    shift_tokens,
    softmax_nll,
    authored_lineage,
)

print("repository root:", ROOT)
print("dataset version:", DATASET_VERSION)
print("training version:", TRAINING_VERSION)
print("scale:\n", SCALE_LIMIT)
print("boundary:\n", TRAINING_TIME_BOUNDARY)


## M30 → M31 boundary: the block in, the training system out

M30 handed a declared pre-norm block with named checkpoints. That block
still sits in the middle of a language-model stack:

`ids → embed → M30 block → unembed → logits`

M31 does **not** train that stack. Teaching scale is a `(V, V)` next-token
score table plus an explicit causal objective. The block can be unembedded
once so you see where logits would attach. Token selection from those
logits is M32.

Canonical sources: `hf-llm-course` and `karpathy-zero-to-hero`.


## Frozen teaching fixtures

Declare the useful whole **before** the first shift.

| Knob | Teaching value |
| --- | --- |
| Corpus | `datasets/M31/corpus.json` (`v07-teaching-corpus-1`) |
| Tokenizer | M27 word scheme `v06.1`, specials on |
| Model | `(V, V)` bigram score table |
| Seed / steps / lr | `3101` / `60` / `0.75` |
| Eval alignment | always correct next-token |
| Leak target | authored eval id reserved in the fixture |

The table is authored so shift and contamination bugs are visible. It is
**not a production** pretrain corpus.


In [ ]:
word = load_tokenizer(scheme="word")
tokenizer = load_teaching_tokenizer()
assert word.version == tokenizer.version
docs = load_teaching_corpus()
lineage = authored_lineage(docs, tokenizer_name=tokenizer.name, tokenizer_version=tokenizer.version)
print("tokenizer", tokenizer.name, tokenizer.version, "V", tokenizer.identity.vocab_size)
print("downloaded", tokenizer.identity.downloaded, "network", tokenizer.identity.network_required)
print("authored train", lineage.train_ids)
print("authored eval", lineage.eval_ids)
print("disjoint", not lineage.contaminated)
print("split hash", lineage.split_hash)
for doc in docs:
    print(doc.authored_split, doc.doc_id, doc.checksum, repr(doc.text))
print("hand shift operands", SHIFT_TOKENS, "->", SHIFT_INPUTS, SHIFT_TARGETS)
print("M30 diagram:\n", PRE_NORM_DIAGRAM)


Train documents repeat a cat/mat pattern plus support phrases. Eval
holds a shared-prefix continuation and a unique document. Authored
splits start disjoint. The M27 tokenizer is the same bundled word
scheme from tokenization; M31 does not invent a new encoding.


## Predict before running — whole training system

Timestamp a prediction before `run-whole`.

The teaching system map prints, then one frozen M30 block is unembedded
to vocab-sized logits. No weights of the block are updated.

Predict:
- whether the unembed output shape is `(batch, seq, V)` or `(seq, seq)`
- whether this cell trains the transformer block
- which side of the **training-time** / inference-time boundary this
  forward pass sits on


In [ ]:
print(SYSTEM_MAP)
raw_block = transformer_block(X_CASH_CONTEXT)
print("raw M30 output shape", raw_block.shapes["output"])
unembed = np.eye(4, 8)
cash_logits, block_trace = frozen_block_logits(X_CASH_CONTEXT, unembed)
print("block convention", block_trace.declared_convention, "defect", block_trace.defect)
print("block output", block_trace.shapes["output"], "logits", cash_logits.shape)
print("block handoff", block_observability(block_trace)["handoff"])
print("training-time boundary:", TRAINING_TIME_BOUNDARY)
assert cash_logits.shape == (1, 3, 8)
assert block_trace.defect == "none"
print("the block produced a residual stream; M31 still has to attach an objective")


The block still maps `(B, T, d_model) → (B, T, d_model)`. Unembed turns
the last axis into vocabulary logits. That is a **shape** fact, not a
trained LLM. Next, build the objective those logits would be scored
against.


## Predict before running — next-token pairs

Timestamp a prediction before `run-pairs`.

Independent tokens `SHIFT_TOKENS = (10, 20, 30, 40)` and document `d01`
after M27 encoding.

Predict:
- the values of `tokens[:-1]` and `tokens[1:]` for the hand sequence
- whether `[BOS]` is an input that must predict the first content token
- how many targets a sequence of length 4 produces


In [ ]:
hand_pairs = shift_tokens(SHIFT_TOKENS)
print("hand tokens", SHIFT_TOKENS)
print("hand inputs", hand_pairs.inputs, "expected", SHIFT_INPUTS)
print("hand targets", hand_pairs.targets, "expected", SHIFT_TARGETS)
print("hand causal", hand_pairs.causal, "n_targets", hand_pairs.n_targets)
d01 = [doc for doc in docs if doc.doc_id == "d01"][0]
d01_enc = tokenizer.encode(d01.text, add_special_tokens=True)
d01_pairs = shift_tokens(d01_enc.ids, token_pieces=d01_enc.tokens)
print("d01 pieces", d01_enc.tokens)
print("d01 inputs pieces", d01_enc.tokens[:-1])
print("d01 target pieces", d01_enc.tokens[1:])
print("d01 n_targets", d01_pairs.n_targets)
assert hand_pairs.inputs == SHIFT_TOKENS[:-1]
assert hand_pairs.targets == SHIFT_TOKENS[1:]
assert d01_pairs.causal


Correct pairs are a slice, not a learned trick: inputs equal
`tokens[:-1]`, targets equal `tokens[1:]`. `[BOS]` is allowed to predict
the first word; the last content token predicts `[EOS]`. If those two
slices ever match each other, the collator is no longer causal.


## Predict before running — tiny causal NLL

Timestamp a prediction before `run-loss`.

Logits `(0.0, 0.0)`, target class `1`. Uniform two-class scores.

Predict:
- the softmax probabilities
- the NLL, compared with `log(2)`
- whether this number needs a neural net


In [ ]:
nll_two = softmax_nll(NLL_LOGITS, NLL_TARGET)
print("logits", NLL_LOGITS, "target", NLL_TARGET)
print("nll", nll_two, "log(2)", math.log(2.0))
print("uniform V=30 would be", math.log(tokenizer.identity.vocab_size))
assert abs(nll_two - math.log(2.0)) < 1e-12
print("causal-LM loss is mean NLL of the true next token under softmax")


Uniform logits over two classes cost `log(2)`. The teaching table starts
near uniform over `V`, so the first train step should sit near `log(V)`.
That is the baseline a later curve has to beat — on the **correct**
targets.


## Predict before running — bounded teaching simulation

Timestamp a prediction before `run-train`.

Same corpus, seed `3101`, 60 SGD steps on the `(V, V)` table. Evaluation
uses correct next-token alignment on authored eval ids.

Predict:
- whether train objective NLL falls below the `log(V)` baseline
- whether that drop proves a general language model
- what the checkpoint's adaptation stage label should be


In [ ]:
healthy = run_causal_pipeline()
print("defect", healthy.defect, "alignment", healthy.alignment)
print("first train", healthy.train_objective_losses[0], "last train", healthy.final_train_objective_loss)
print("last eval true NLL", healthy.final_eval_true_loss)
print("unseen pair NLL", healthy.unseen_pair_nll)
print("checkpoint", healthy.checkpoint.checkpoint_id)
print("stage", healthy.checkpoint.adaptation_stage, "inference_ready", healthy.checkpoint.inference_ready)
print("training_time", healthy.checkpoint.training_time)
assert healthy.final_train_objective_loss < healthy.train_objective_losses[0]
assert healthy.checkpoint.inference_ready and not healthy.checkpoint.training_time
print(SCALE_LIMIT)


Train NLL fell on eight synthetic documents. The checkpoint is labeled
`pretrained` at **teaching scale**, then marked inference-ready with
training-time false. That is a produced toy table, not a foundation
model. Next, ask whether held-out text followed.


## Predict before running — memorization vs held-out

Timestamp a prediction before `run-heldout`.

Invariant: same table, seed, and budget as `run-train`. Change: look at
authored eval documents, including a shared-prefix continuation.

Predict:
- whether eval true NLL is below train NLL
- whether unseen bigrams can stay near the uniform baseline
- why train loss alone is weak evidence


In [ ]:
print("train ids", healthy.authored_lineage.train_ids)
print("eval ids", healthy.authored_lineage.eval_ids)
print("train objective", healthy.final_train_objective_loss)
print("eval true next-token", healthy.final_eval_true_loss)
print("unseen pair NLL", healthy.unseen_pair_nll)
e01 = [doc for doc in docs if doc.doc_id == "e01"][0]
e01_enc = tokenizer.encode(e01.text, add_special_tokens=True)
print("e01 pieces", e01_enc.tokens)
assert healthy.final_eval_true_loss > healthy.final_train_objective_loss
assert healthy.unseen_pair_nll > healthy.final_train_objective_loss
print("low train NLL did not make the held-out continuation cheap")


The shared prefix is familiar; the last continuation is not a train
bigram. Unseen-pair NLL stays high. Training objective quality is not
downstream task quality, and it is not a license to skip a protected
eval split.


## Predict before running — target alignment

Timestamp a prediction before `run-shift`.

Same tokens as the hand fixture and `d01`. Compare correct next-token
alignment with a one-position-wrong collator. No training in this cell.

Predict:
- which slice equals the correct targets
- which constructed field would move if the collator stopped shifting
- whether `n_targets` must change for a pure alignment bug


In [ ]:
correct = shift_tokens(SHIFT_TOKENS, alignment="correct")
wrong = shift_tokens(SHIFT_TOKENS, alignment="target_shift_wrong")
print("correct inputs", correct.inputs)
print("correct targets", correct.targets)
print("wrong targets", wrong.targets)
print("wrong causal", wrong.causal)
print("n_targets same", correct.n_targets == wrong.n_targets)
d01_wrong = shift_tokens(d01_enc.ids, alignment="target_shift_wrong", token_pieces=d01_enc.tokens)
print("d01 wrong target pieces", d01_enc.tokens[:-1])
assert correct.targets == SHIFT_TOKENS[1:]
assert wrong.targets == SHIFT_TOKENS[:-1]
assert correct.n_targets == wrong.n_targets
print("alignment can be wrong while the tensor still looks like a batch")


A one-position error keeps the same number of rows and can still look
like a training batch. The discriminator is the constructed target
slice, not a learning-rate search. The next experiments keep this
tokenizer and corpus fixed.


## Predict before running — context length

Timestamp a prediction before `run-context`.

Invariant: tokenizer and `d01` / `CONTEXT_SEQUENCE` texts stay fixed.
Change: the window length.

Predict:
- `n_targets` for length-5 tokens with `C=5` and `C=3`
- whether a right-side suffix can leave the objective
- whether this is a decoding budget (it is not; that is M32)


In [ ]:
print("independent sequence", CONTEXT_SEQUENCE)
for row in context_length_effect(CONTEXT_SEQUENCE, (5, 3, 2)):
    print(row)
print("formula C=3", n_prediction_targets(5, 3))
d01_short = shift_tokens(d01_enc.ids, context_length=SHORT_CONTEXT_LENGTH, token_pieces=d01_enc.tokens)
print("d01 full n_targets", d01_pairs.n_targets, "short", d01_short.n_targets)
print("short window pieces", d01_enc.tokens[:SHORT_CONTEXT_LENGTH])
print("dropped pieces", d01_enc.tokens[SHORT_CONTEXT_LENGTH:])
assert n_prediction_targets(5, 3) == 2
assert d01_short.n_targets == SHORT_CONTEXT_LENGTH - 1
print("context length here is a data-shaping control, not a sampler")


Shorter `C` means fewer prediction targets and a dropped suffix. The
tokenizer did not change. Do not confuse this window with an inference
max-token budget; M31 is still building the training objective.


## Predict before running — contamination

Timestamp a prediction before `run-contamination`.

Invariant: evaluation still uses correct shift on the **authored** eval
ids. Change: copy protected eval document `e02` into the train id list
and retrain.

Predict:
- whether `train ∩ eval` stays empty
- whether held-out NLL is still protected evidence
- what a lineage report should flag without touching learning rate


In [ ]:
leaked_lineage = lineage_with_leak(lineage)
print(lineage_report(leaked_lineage))
leaked = run_causal_pipeline(defect="held_out_leak")
print("used train", leaked.used_lineage.train_ids)
print("authored eval", leaked.authored_lineage.eval_ids)
print("healthy unseen", healthy.unseen_pair_nll, "leaked unseen", leaked.unseen_pair_nll)
print("healthy eval", healthy.final_eval_true_loss, "leaked eval", leaked.final_eval_true_loss)
assert leaked.used_lineage.contaminated
assert leaked.used_lineage.eval_ids == lineage.eval_ids
assert leaked.unseen_pair_nll < healthy.unseen_pair_nll
print("the eval formula did not change; the split did")


Overlap is the definition of contamination. Eval NLL can fall because
the document is no longer held out. That is invalid evidence, not a
better model. Repair the boundary before you celebrate the curve.


## Predict before running — stage classification

Timestamp a prediction before `run-stages`.

No weights change. Classify frozen interventions against the stage
definitions.

Predict labels for:
- next-token on unlabeled text
- instruction pairs
- preference / RLHF-style post-training
- scoring a protected split
- generating from a frozen checkpoint
- changing a prompt without updating weights


In [ ]:
for name, definition in STAGE_DEFINITIONS.items():
    print(name, ":", definition)
for intervention in (
    "next_token_on_unlabeled_corpus",
    "instruction_supervised_pairs",
    "preference_ranking_or_rlhf",
    "score_protected_held_out",
    "generate_from_frozen_checkpoint",
    "change_prompt_without_weight_update",
):
    print(intervention, "->", classify_intervention(intervention))
print("inference is not a training update:", TRAINING_TIME_BOUNDARY)


Pretraining is unlabeled next-token work. Adaptation and post-training
still update weights, under different objectives. Evaluation scores a
frozen split. Inference consumes a produced checkpoint; how it selects
tokens is M32. Prompt changes without weight updates are not pretraining.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(healthy.train_objective_losses, label="healthy train objective")
ax.plot(healthy.eval_true_losses, label="healthy eval true NLL")
ax.plot(leaked.eval_true_losses, label="contaminated eval true NLL")
ax.set_xlabel("step")
ax.set_ylabel("mean NLL")
ax.set_title("Teaching-scale causal NLL (not a production run)")
ax.legend()
fig.tight_layout()
plt.show()
print("plot is evidence of curves, not of a general LLM")


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `shift_tokens`, `run_causal_pipeline`, `lineage_with_leak`,
`repair_run`, and `make_checkpoint` in
`missions/M31/llm_training_core.py`.

Predict:
- the `(input, target)` pairs for a four-token window `(10, 20, 30, 40)`
- whether evaluation uses the defective alignment or the correct one
- which lineage field moves first if a protected eval id is leaked
- what M32 is allowed to do with `inference_ready=True`

Do not search the file for a sampler or a retrieval index.


In [ ]:
pipe_src = inspect.getsource(run_causal_pipeline)
shift_src = inspect.getsource(shift_tokens)
repair_src = inspect.getsource(repair_run)
print("shift uses window[1:] for correct targets", "window[1:]" in shift_src)
print("pipeline eval alignment is correct", 'alignment="correct"' in pipe_src)
print("eval uses authored.eval_ids", "authored.eval_ids" in pipe_src)
print("repair recomputes from trace.documents", "trace.documents" in repair_src)
print("repair docstring", (repair_run.__doc__ or "").strip().splitlines()[0])
print("hand four-token pairs", shift_tokens((10, 20, 30, 40)).inputs, shift_tokens((10, 20, 30, 40)).targets)
report = observability_report(healthy)
print("handoff", report["handoff"])
assert "authored.eval_ids" in pipe_src
assert "trace.documents" in repair_src
print("tokenize, window, shift, batch, NLL, split, checkpoint, eval — then stop")


## Predict before running — Controlled failure: target shift

Timestamp a prediction before `run-failure-shift`.

Train the teaching table with one named collator defect. Seed, steps,
corpus texts, and learning rate stay fixed. Evaluation still scores
true next-token NLL.

Predict:
- which constructed field would disagree with the next token
- whether training **objective** loss can still fall
- whether you should reach for a new learning rate first


In [ ]:
broken_shift = pipeline_with_defect(defect="target_shift_wrong")
print("defect", broken_shift.defect)
print("first divergence vs healthy", first_divergence(healthy, broken_shift))
print("sample window", broken_shift.windows[0].window)
print("sample inputs", broken_shift.windows[0].inputs)
print("sample targets", broken_shift.windows[0].targets)
print("causal?", broken_shift.windows[0].causal)
print("objective last", broken_shift.final_train_objective_loss, "first", broken_shift.train_objective_losses[0])
print("eval true last", broken_shift.final_eval_true_loss)
assert first_divergence(healthy, broken_shift) == "targets"
assert broken_shift.final_train_objective_loss < broken_shift.train_objective_losses[0]
print("loss fell on the objective the collator actually trained")


## Predict before running — Controlled failure: held-out leak

Timestamp a prediction before `run-failure-leak`.

One named lineage defect. The eval id list and NLL formula stay the
same. A protected eval document is also used as training material.

Predict:
- which lineage field should become non-empty
- whether eval true NLL can fall without a better objective
- what you inspect before hyperparameters


In [ ]:
broken_leak = pipeline_with_defect(defect="held_out_leak")
print("defect", broken_leak.defect)
print("first divergence vs healthy", first_divergence(healthy, broken_leak))
print(lineage_report(broken_leak.used_lineage))
print("windows still causal", all(window.causal for window in broken_leak.windows))
print("eval true", broken_leak.final_eval_true_loss, "healthy eval", healthy.final_eval_true_loss)
assert first_divergence(healthy, broken_leak) == "train_doc_ids"
assert broken_leak.used_lineage.contaminated
print("the score function did not change; membership did")


## Diagnosis record (your log, not this repo)

Fill symptom, plausible hypotheses, discriminating experiment, observed
result, root cause, smallest repair, verification, and regression
evidence in your evidence log.

Discriminators live in constructed pairs and split overlap. Do not
start with a new optimizer.


## Predict before running — repair from the broken traces

Timestamp a prediction before `run-failure-repair`.

`repair_run` must reuse each broken trace's documents, seed, steps,
context length, and learning rate, with the named defect cleared.

Predict:
- whether repaired windows are causal
- whether repaired train ids match the authored split
- whether the broken objects still diverge after repair


In [ ]:
repaired_shift = repair_run(broken_shift)
repaired_leak = repair_run(broken_leak)
print("repaired shift defect", repaired_shift.defect, "causal", all(w.causal for w in repaired_shift.windows))
print("divergence broken_shift vs repaired", first_divergence(broken_shift, repaired_shift))
print("repaired leak contaminated", repaired_leak.used_lineage.contaminated)
print("divergence broken_leak vs repaired", first_divergence(broken_leak, repaired_leak))
print("broken shift still not causal", any(not w.causal for w in broken_shift.windows))
print("broken leak still has overlap", broken_leak.used_lineage.contaminated)
assert repaired_shift.seed == broken_shift.seed
assert repaired_shift.documents == broken_shift.documents
assert first_divergence(broken_shift, repaired_shift) == "targets"
assert first_divergence(broken_leak, repaired_leak) == "train_doc_ids"
print("repair restored the boundary; the defective objects remain as regression fixtures")


Repair did not invent a second unrelated happy-path run from scratch
configuration. It reused the broken traces. The defective objects still
show the first divergences — that is the regression evidence.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- constructed next-token pairs for a teaching sequence
- independent softmax NLL on the two-class fixture
- train versus held-out true next-token NLL
- lineage overlap under contamination
- stage labels for the frozen interventions
- shift/leak diagnosis and `repair_run`

See `missions/M31/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M31/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Build next-token targets for a fresh id sequence, compute a three-class
NLL, draw the training lifecycle, spot a shift or a leak, and explain
why low training loss is weak evidence.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M31/adr_prompt.md` to choose a V07 training-data and
evaluation provenance policy (dataset version, split lineage, checkpoint
identity, adaptation stage, audit metadata). Do not claim a production
LLM.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M31 for the package, not as a substitute for the learner ADR.


## M31 → M32 handoff

M30 supplied a block. M31 attached a causal next-token objective,
explicit pair construction, lineage, and a stage-aware checkpoint.

M32 may consume `inference_ready` checkpoints and control **token
selection**. It must not relabel a sampling change as a training-stage
change, and it must not mix eval documents into train.

Reusable artifacts: `StageAwareCheckpoint`, authored versus used
lineage, objective `causal_next_token`, version `v07-teaching-lm-1`,
and the training-time versus inference-time boundary.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why must targets equal `tokens[1:]` rather than `tokens[:-1]`?
2. What identity tells you an evaluation split is still protected?
3. Why is a falling training curve insufficient evidence?
4. What must M32 receive that a weight matrix without lineage cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert hand_pairs.inputs == SHIFT_TOKENS[:-1]
assert hand_pairs.targets == SHIFT_TOKENS[1:]
assert abs(nll_two - math.log(2.0)) < 1e-12
assert n_prediction_targets(5, 3) == 2
assert healthy.defect == "none" and all(w.causal for w in healthy.windows)
assert healthy.final_train_objective_loss < healthy.train_objective_losses[0]
assert first_divergence(healthy, broken_shift) == "targets"
assert first_divergence(healthy, broken_leak) == "train_doc_ids"
assert first_divergence(broken_shift, repaired_shift) == "targets"
assert first_divergence(broken_leak, repaired_leak) == "train_doc_ids"
assert broken_shift.windows[0].targets == broken_shift.windows[0].window[:-1]
assert LEAK_DOC_ID in broken_leak.used_lineage.train_ids
assert not repaired_leak.used_lineage.contaminated
assert cash_logits.shape[-1] == 8
assert classify_intervention("generate_from_frozen_checkpoint") == "inference"
print("M31 integrity checks passed")
